# Fiber collections

A `FiberCollection` is detached geometry that can be generated, loaded,
transformed, combined, and later inserted into a recipe. It is allowed
to contain overlaps; relaxation happens after insertion.

In [ ]:
import tangle
from tangle.units import mm, um

material = tangle.Material("fiber", diameter=10 * um)
# Collection coordinates are local until Recipe.insert applies a rigid
# transform. Overlap is allowed at this stage.
ply = tangle.FiberCollection("ply 2")
fiber_index = ply.add_fiber(
    [[0.0, 0.0, 0.0], [0.5 * mm, 0.0, 0.0]],
    material,
    rest_centerline=[[0.0, 0.0, 0.0], [0.5 * mm, 0.0, 0.0]],
    # Tags remain descriptive metadata; formation_layer participates in
    # later layer-aware recipe operations.
    tags={"family": "machine-direction", "source": "measured"},
    formation_layer=2,
)
print(fiber_index, len(ply), ply.layer_ids())

## Construction and selection operations

- `add_fiber(...)` controls placed/rest centerlines, material, tags, and layer.
- `from_centerlines(...)` assigns one material and optional layer in bulk.
- `a + b` returns a new collection containing both; `a.extend(b)`
  appends `b` to `a` in place.
- `layer_ids()` lists the formation layers present.
- `select_layer(...)` creates a collection containing one formation
  layer. A missing layer raises `ValueError` unless `allow_empty=True`.
- `centerlines()` and `rest_centerlines()` return ordinary Python lists.

In [ ]:
# Bulk construction is convenient when one material/layer applies to
# many imported centerlines.
transverse = tangle.FiberCollection.from_centerlines(
    [[[0.0, 0.0, 0.0], [0.0, 0.5 * mm, 0.0]]],
    material,
    name="transverse",
    formation_layer=3,
)
# `+` builds a new collection and leaves both operands unchanged.
all_fibers = ply + transverse
all_fibers.name = "two plies"
# extend() appends in place when mutating a collection is intended.
growing = tangle.FiberCollection("growing")
growing.extend(ply)
# Selection returns another detached collection; it does not mutate the
# combined source collection.
selected = all_fibers.select_layer(2, name="only ply 2")
missing = all_fibers.select_layer(7, allow_empty=True)
print(len(all_fibers), len(ply), len(growing), len(selected), len(missing))
print(all_fibers.layer_ids())